In [2]:
import scipy.io as sio
import numpy as np
import pandas as pd
import os
from scipy.stats import skew, kurtosis
from scipy.signal import welch
from scipy.signal import butter, filtfilt
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
import torch.nn.functional as F
from torch.utils.data import Dataset

import hdf5storage
from tqdm import tqdm

In [22]:
import time

In [3]:
import warnings
warnings.filterwarnings("ignore")

import mne
mne.set_log_level("ERROR")


Data Loading

In [4]:
def load_dreamer(path):
    mat = hdf5storage.loadmat(path)

    all_subjects = []
    all_labels = []
    all_baselines = []

    DREAMER = mat['DREAMER'][0, 0]
    data = DREAMER['Data'][0]            # list of subjects

    for subj in data:
        EEG = subj['EEG'][0, 0]

        # Same as your original code
        val_raw = subj['ScoreValence'][0][0]   # 18x1 or 1x18
        aro_raw = subj['ScoreArousal'][0][0]

        # Flatten to 1D arrays (length 18)
        val = np.array(val_raw).reshape(-1)
        aro = np.array(aro_raw).reshape(-1)

        subject_trials = []
        subject_labels = []
        subject_baselines = []
        # 18 trials per subject (or len(val) just in case)
        n_trials = len(val)

        for i in range(n_trials):
            stim = EEG['stimuli'][0, 0][i, 0]   # n × 14 (time × channels)
            if stim.size == 0:
                continue
            baseline = EEG['baseline'][0, 0][i, 0] # m × 14
            # Convert to channels × time to match GAMEEMO / DENS
            trial = stim.T.astype(np.float32)   # 14 × n
            baseline = baseline.T.astype(np.float32) # 14 × m
            subject_trials.append(trial)
            subject_labels.append((float(val[i]), float(aro[i])))
            subject_baselines.append(baseline)

        all_subjects.append(subject_trials)
        all_labels.append(subject_labels)
        all_baselines.append(subject_baselines)

    return all_subjects, all_labels, all_baselines

In [5]:
# Load GAMEEMO
def load_gameemo_ratings(ratings_path):
        # Load SAM ratings
    df = pd.read_csv(ratings_path)

    ratings_dict = {}
    for _, row in df.iterrows():
        ratings_dict[(row.subject, row.game)] = (row.valence, row.arousal)

    return ratings_dict


def load_gameemo_mat(mat_path):
    GAMEEMO_CHANNELS = ["AF3", "AF4", "F3", "F4", "F7", "F8", "FC5", "FC6", "O1", "O2", "P7", "P8", "T7", "T8"]
    mat = sio.loadmat(mat_path)

    eeg_list = []
    for ch in GAMEEMO_CHANNELS:
        if ch not in mat:
            raise ValueError(f"Missing channel {ch} in {mat_path}")
        eeg_list.append(mat[ch].squeeze())  # shape (samples,)

    eeg = np.stack(eeg_list, axis=0)  # shape = (14 channels, samples)

    return eeg
def load_gameemo_subject(subj_dir, subj_id, ratings_dict):

    eeg_data = []
    labels = []

    for game in [1, 2, 3, 4]:
        val, aro = ratings_dict.get((subj_id, game), (None, None))

        if val is None or aro is None or pd.isna(val) or pd.isna(aro):
            print(f"[SKIP] Missing SAM for {subj_id} G{game}")
            continue

        mat_path = os.path.join(
            subj_dir,
            "Preprocessed EEG Data",
            ".mat format",
            f"{subj_id}G{game}AllChannels.mat"
        )

        if not os.path.exists(mat_path):
            print(f"[SKIP] Missing EEG: {mat_path}")
            continue

        try:
            eeg = load_gameemo_mat(mat_path)
        except Exception as e:
            print(f"[SKIP] Can't load EEG for {subj_id} G{game}: {e}")
            continue

        eeg_data.append(eeg)
        labels.append((val, aro))

    return eeg_data, labels


In [6]:
# Load DENS

# ---- Channel mapping (14-channel subset) ----
dens_mapping = ["E17","E8","E22","E9","E27","E6","E33","E4",
                "E70","E83","E51","E92","E47","E98"]


# ---- Preprocess one DENS trial (segment->downsample->select 14 channels) ----
def preprocess_dens_epoch(raw, start_samp, end_samp):

    # Extract 128-ch segment
    data = raw.get_data(start=start_samp, stop=end_samp)  # shape (128, T)

    # Create temporary raw object so MNE can resample
    info = mne.create_info(
        ch_names=[f"E{i+1}" for i in range(data.shape[0])],
        sfreq=raw.info["sfreq"],
        ch_types="eeg"
    )

    raw_epoch = mne.io.RawArray(data, info)

    # ↓↓↓ DOWN-SAMPLE TO 128 Hz ↓↓↓
    raw_epoch.resample(128)
    data_ds = raw_epoch.get_data()  # shape (128, T_down)

    # ↓↓↓ SELECT 14 CHANNELS MATCHING GAMEEMO/DREAMER ↓↓↓
    idxs = [raw_epoch.ch_names.index(ch) for ch in dens_mapping]
    data_14 = data_ds[idxs, :]     # shape (14, T_down)

    return data_14

def load_eeglab_safe(path): 
    """ Attempt to load an EEGLAB .set file using several fallback strategies. Works for both v7.3 (HDF5) and standard EEGLAB files. """ 
    #print(f"Loading EEGLAB file: {path}") 
    flag = 0
    # Method 1: Standard load 
    try: 
        #print("Trying: preload=True ...") 
        return mne.io.read_raw_eeglab(path, preload=True) 
    except Exception as e1: 
        flag = 1
        #print("Method 1 failed:", e1) 
    
    # Method 2: Load without preload, then load data 
    try: 
        #print("Trying: preload=False then load_data() ...") 
        raw = mne.io.read_raw_eeglab(path, preload=False) 
        raw.load_data() 
        return raw 
    except Exception as e2: 
        flag = 2
        #print("Method 2 failed:", e2) 
    
    # Method 3: Using uint16_codec workaround (needed for v7.3) 
    try: 
        #print("Trying: preload=True with uint16_codec='utf-8' ...") 
        return mne.io.read_raw_eeglab(path, preload=True, uint16_codec="utf-8") 
    except Exception as e3: 
        flag = 3
        #print("Method 3 failed:", e3) 
        raise RuntimeError("❌ Failed to load .set file using all available methods.")

# ---- DENS Loader ----
def load_dens(root_dir):

    subject_dirs = sorted([s for s in os.listdir(root_dir) if s.startswith("sub-mit")])

    all_subjects = []
    all_labels = []

    for subj in tqdm(subject_dirs, disable=True):
        subj_path = os.path.join(root_dir, subj)
        #print(f"\nProcessing subject: {subj}")

        beh_file = os.path.join(subj_path, "beh", f"{subj}_task-Emotion_beh.tsv")
        eeg_file = os.path.join(subj_path, "eeg", f"{subj}_task-Emotion_eeg.set")
        events_file = os.path.join(subj_path, "eeg", f"{subj}_task-emotion_events.tsv")

        # ---- Skip incomplete .fdt files ----
        fdt_file = eeg_file.replace("_eeg.set", "_eeg.fdt")
        if not os.path.exists(fdt_file):
            #print(f"[SKIP] {subj}: missing FDT")
            continue

        fdt_size = os.path.getsize(fdt_file) / 1024**2
        if fdt_size < 100:
            #print(f"[SKIP] {subj}: FDT too small ({fdt_size:.1f} MB)")
            continue

        if not (os.path.exists(beh_file) and os.path.exists(events_file)):
            #print(f"[SKIP] {subj}: missing beh or events")
            continue

        # ---- Load behavior ----
        beh = pd.read_csv(beh_file, sep="\t")
        valences = beh["valence"].values
        arousals = beh["arousal"].values

        # ---- Load events ----
        df_events = pd.read_csv(events_file, sep="\t")
        stm_onsets = df_events[df_events["trial_type"] == "stm"]["onset"].values
        vlnc_onsets = df_events[df_events["trial_type"] == "vlnc"]["onset"].values

        n_trials = min(len(stm_onsets), len(vlnc_onsets), len(valences), len(arousals))
        if n_trials == 0:
            #print(f"[SKIP] {subj}: no usable trials")
            continue

        # ---- Load EEG ----
        try:
            raw = load_eeglab_safe(eeg_file)
        except:
            #print(f"[SKIP] {subj}: failed to load EEG")
            continue

        sfreq = raw.info["sfreq"]    # usually 250 Hz

        subject_data = []
        subject_lab = []

        # ---- Segment each trial ----
        for i in range(n_trials):
            start_t = stm_onsets[i]   # DENS uses seconds, NOT ms
            end_t   = vlnc_onsets[i]

            if end_t <= start_t:
                continue

            start_samp = int(start_t)
            end_samp   = int(end_t)


            if end_samp - start_samp < sfreq:
                continue  # skip <1s segment

            # ↓↓↓ preprocess (cut→downsample→select 14 channels)
            epoch_14 = preprocess_dens_epoch(raw, start_samp, end_samp)

            subject_data.append(epoch_14)
            subject_lab.append((float(valences[i]), float(arousals[i])))

        # ---- Skip empty subjects ----
        if len(subject_data) == 0:
            #print(f"[SKIP] {subj}: no valid clips after processing")
            continue

        all_subjects.append(subject_data)
        all_labels.append(subject_lab)

    return all_subjects, all_labels


In [7]:
DREAMER_subjects, DREAMER_labels, DREAMER_baselines = load_dreamer("C:/Users/Yebbet/Desktop/School/APS360/EEG_Datasets/DREAMER.mat")


In [7]:
DREAMER_subjects[0][0].shape

(14, 25472)

In [8]:
root_dir = "C:/Users/Yebbet/Desktop/School/APS360/EEG_Datasets/GAMEEMO"
subject_gamedirs = sorted([s for s in os.listdir(root_dir) if s.startswith("(") and s.endswith(")")])
GAMEEMO_subjects = []
GAMEEMO_labels = []
ratings_dict = load_gameemo_ratings("C:/Users/Yebbet/Desktop/School/APS360/EEG_Datasets/GAMEEMO/gameemo_ratings.csv")
for subj_dir in subject_gamedirs:
    subj_path = os.path.join(root_dir, subj_dir)
    subj_id = subj_dir.strip("()")
    eeg_data, labels = load_gameemo_subject(subj_path, subj_id, ratings_dict)
    GAMEEMO_subjects.append(eeg_data)
    GAMEEMO_labels.append(labels)

[SKIP] Missing SAM for S19 G1


In [9]:
DENS_subjects, DENS_labels = load_dens("C:/Users/Yebbet/Desktop/School/APS360/EEG_Datasets/ds003751/")

In [31]:
type(DENS_labels[0])
print(DENS_labels[0])

[(8.97, 7.95), (9.0, 9.0), (7.01, 6.0), (1.05, 7.1), (1.0, 7.95), (4.42, 7.53), (1.0, 8.03), (6.06, 7.01), (3.07, 6.87), (1.0, 9.0), (1.0, 8.86)]


Preprocessing

In [88]:
# -------------------------
# Basic helpers
# -------------------------

def bandpass(data, low=0.5, high=45, fs=128, order=5):
    """Bandpass filter along time axis."""
    b, a = butter(order, [low/(fs/2), high/(fs/2)], btype='band')
    return filtfilt(b, a, data, axis=1)

def reref_avg(data):
    """Average reference."""
    avg = data.mean(axis=0, keepdims=True)
    return data - avg

def zscore(data):
    """Z-score per channel."""
    mean = data.mean(axis=1, keepdims=True)
    std = data.std(axis=1, keepdims=True) + 1e-6
    return (data - mean) / std

def pad_or_crop(data, target_len):
    """Pad or crop trial to fixed length."""
    C, T = data.shape
    if T > target_len:
        return data[:, :target_len]
    else:
        pad = target_len - T
        return np.pad(data, ((0,0),(0,pad)), mode="constant")


# -----------------------------------------------------
# A unified function for all 3 dataset types
# -----------------------------------------------------

def preprocess_eeg_trial(
    trial, 
    dataset,              # "DREAMER", "GAMEEMO", "DENS"
    baseline=None,        # only used for DREAMER
    fs=128,
    target_len=512
):
    """
    Preprocess a single 14×T EEG trial depending on dataset type.
    Returns: preprocessed trial of shape (14, target_len)
    """

    # Ensure float32
    trial = trial.astype(np.float32)

    # -------------------------
    # DREAMER-specific step
    # -------------------------
    if dataset.upper() == "DREAMER":
        # DREAMER is already 128 Hz and already raw
        # Standard processing (except DO NOT double-bandpass if baseline already filtered; DREAMER is raw → SAFE TO FILTER)
        if trial.shape[1] < 40:
            pass
        elif baseline is not None:
            trial = trial - baseline.mean(axis=1, keepdims=True)

        # 14 channels at 128 Hz, safe to filter
        trial = bandpass(trial, fs=fs)
        trial = reref_avg(trial)
        trial = np.clip(trial, -3, 3)
    # -------------------------
    # GAMEEMO-specific step
    # -------------------------
    elif dataset.upper() == "GAMEEMO":
        # GAMEEMO's folder is "Preprocessed EEG Data" = already filtered in the publication pipeline
        # Do NOT run bandpass or average ref again.
        # ONLY normalize + pad.
        pass

    # -------------------------
    # DENS-specific step
    # -------------------------
    elif dataset.upper() == "DENS":
        # You already performed:
        #   - segmentation
        #   - resampling 250 → 128
        #   - channel mapping 128 → 14
        # So apply full standard pipeline
        trial = bandpass(trial, fs=fs)
        trial = reref_avg(trial)

    # -------------------------
    # Shared final steps
    # -------------------------

    # Z-score normalization
    trial = zscore(trial)
    #print(trial.shape)
    # Pad / crop
    trial = pad_or_crop(trial, target_len)

    return trial.astype(np.float32)


In [117]:
# DREAMER preprocess
for i in range(len(DREAMER_subjects)):
    for j in range(len(DREAMER_subjects[i])):
        stim = DREAMER_subjects[i][j]  # time × channels
        base = DREAMER_baselines[i][j] # Dummy baseline, replace with actual if available
        DREAMER_subjects[i][j] = preprocess_eeg_trial(
            stim,
            dataset="DREAMER",
            baseline=base,
            fs=128,
            target_len=512
        )

In [118]:
# GAMEEMO preprocess
for i in range(len(GAMEEMO_subjects)):
    for j in range(len(GAMEEMO_subjects[i])):
        eeg = GAMEEMO_subjects[i][j]
        GAMEEMO_subjects[i][j] = preprocess_eeg_trial(
            eeg,
            dataset="GAMEEMO",
            baseline=None,
            fs=128,
            target_len=512
        )

In [119]:
# DENS preprocess
for i in range(len(DENS_subjects)):
    for j in range(len(DENS_subjects[i])):
        eeg = DENS_subjects[i][j]
        DENS_subjects[i][j] = preprocess_eeg_trial(
            eeg,
            dataset="DENS",
            baseline=None,
            fs=128,
            target_len=512
        )

Combine Datasets

In [120]:
all_eeg = []
all_labels = []
all_source = []   # optional: to track dataset origin
all_groups = []      # <-- NEW

global_subject_id = 0   # will increment across datasets


def append_dataset(eeg_list, label_list, source_name):
    """
    eeg_list    : list of subjects → each subject is list of trials
    label_list  : list of subjects → each subject is list of labels
    source_name : string ("DREAMER" / "GAMEEMO" / "DENS")
    """
    global global_subject_id

    for subj_eeg, subj_labels in zip(eeg_list, label_list):

        # For each subject → add trials
        for eeg, lab in zip(subj_eeg, subj_labels):

            if eeg is None or lab is None:
                continue

            all_eeg.append(eeg)
            all_labels.append(lab)
            all_source.append(source_name)

            # ★ Track grouping for LOSO / GroupKFold
            all_groups.append(global_subject_id)

        # Done processing 1 subject → move to next subject ID
        global_subject_id += 1



In [121]:
append_dataset(DREAMER_subjects, DREAMER_labels, "DREAMER")
append_dataset(GAMEEMO_subjects, GAMEEMO_labels, "GAMEEMO")
append_dataset(DENS_subjects, DENS_labels, "DENS")


In [110]:
print("Total samples:", len(all_eeg))
print("DREAMER samples:", all_source.count("DREAMER"))
print("GAMEEMO samples:", all_source.count("GAMEEMO"))
print("DENS samples:", all_source.count("DENS"))


Total samples: 622
DREAMER samples: 414
GAMEEMO samples: 111
DENS samples: 97


Binarize labels for classification

In [111]:
def binarize_valence_arousal(all_labels, all_groups):
    all_labels = np.array(all_labels, dtype=float)
    all_groups = np.array(all_groups, dtype=int)

    labels_bin = np.zeros((len(all_labels), 2), dtype=np.int64)

    for subj in np.unique(all_groups):
        idx = np.where(all_groups == subj)[0]

        vals = all_labels[idx, 0]
        aros = all_labels[idx, 1]

        v_thr = np.median(vals)
        a_thr = np.median(aros)

        labels_bin[idx, 0] = (vals >= v_thr).astype(int)
        labels_bin[idx, 1] = (aros >= a_thr).astype(int)

    return labels_bin


In [122]:
all_labels_bin = binarize_valence_arousal(all_labels, all_groups)


Split data

In [123]:
X = np.array(all_eeg, dtype=np.float32)      # (N, 14, 512)
y_bin = np.array(all_labels_bin, dtype=np.int32)  # (N, 2) for analysis if needed
y = np.array(all_labels_bin, dtype=np.int64)     # (N,) 4-class targets for classification
groups = np.array(all_groups, dtype=np.int32)  # (N,)


In [69]:
v = all_labels_bin[:, 0]
a = all_labels_bin[:, 1]

print("Valence distribution:", np.bincount(v))
print("Arousal distribution:", np.bincount(a))


Valence distribution: [244 378]
Arousal distribution: [237 385]


In [124]:
unique_subj = np.unique(groups)

# First split: train subjects vs temp subjects
train_subj, temp_subj = train_test_split(
    unique_subj, test_size=0.30, random_state=42
)

# Second split: val vs test
val_subj, test_subj = train_test_split(
    temp_subj, test_size=0.50, random_state=42
)

print("Train subjects:", train_subj)
print("Val subjects:", val_subj)
print("Test subjects:", test_subj)

train_idx = np.isin(groups, train_subj)
val_idx   = np.isin(groups, val_subj)
test_idx  = np.isin(groups, test_subj)

X_train, y_train = X[train_idx], y[train_idx]
X_val,   y_val   = X[val_idx],   y[val_idx]
X_test,  y_test  = X[test_idx],  y[test_idx]


Train subjects: [40  4 43 19 34 58 25 56 15 27  9 30 26 16 24 55 11 32 53 41 37 29 44  1
 21  2 47 39 35 23 49 10 22 18 59 20  7 42 14 28 51 38]
Val subjects: [36 57  6 13 31 48 46 52 33]
Test subjects: [ 0  5 12 54 45  3  8 17 50]


Create Pytorch Dataset

In [23]:
class EEGDataset(Dataset):
    def __init__(self, X, y_bin):
        """
        X     : (N, 14, 512)
        y_bin : (N, 2) integers in {0,1}
        """
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y_bin, dtype=np.int64)

        assert self.X.ndim == 3, f"X must be (N,14,T). Got {self.X.shape}"
        assert self.y.ndim == 2 and self.y.shape[1] == 2, f"y must be (N,2). Got {self.y.shape}"

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        x = torch.tensor(self.X[idx]).unsqueeze(0)   # (1,14,512)
        y = torch.tensor(self.y[idx])               # (2,)
        return x, y


In [126]:
train_dataset = EEGDataset(X_train, y_train)
val_dataset   = EEGDataset(X_val, y_val)
test_dataset  = EEGDataset(X_test, y_test)


## Primary Model

EEGNet

In [ ]:
class EEGNet(nn.Module):
    def __init__(
        self,
        n_channels=14,
        n_samples=512,
        F1=8,
        D=2,
        F2=16,
        kernel_length=64,
        pool_size=4,
        dropout=0.5,
    ):
        super().__init__()

        # Block 1: Temporal Convolution
        self.firstconv = nn.Sequential(
            nn.Conv2d(
                1, F1,
                kernel_size=(1, kernel_length),
                padding=(0, kernel_length // 2),
                bias=False
            ),
            nn.BatchNorm2d(F1),
        )

        # Block 2: Depthwise spatial convolution
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                F1, F1 * D,
                kernel_size=(n_channels, 1),
                groups=F1,
                bias=False
            ),
            nn.BatchNorm2d(F1 * D),
            nn.ELU(),
            nn.AvgPool2d((1, pool_size)),
            nn.Dropout(dropout),
        )

        # Block 3: Separable conv
        self.separable = nn.Sequential(
            nn.Conv2d(
                F1 * D, F1 * D,
                kernel_size=(1, 16),
                groups=F1 * D,
                padding=(0, 8),
                bias=False
            ),
            nn.Conv2d(
                F1 * D, F2,
                kernel_size=(1, 1),
                bias=False
            ),
            nn.BatchNorm2d(F2),
            nn.ELU(),
            nn.AvgPool2d((1, pool_size)),
            nn.Dropout(dropout),
        )

        # Compute final feature dimension
        with torch.no_grad():
            dummy = torch.zeros(1, 1, n_channels, n_samples)
            feat = self.separable(self.depthwise(self.firstconv(dummy)))
            self._feat_dim = feat.numel()

        # --- TWO HEADS ---
        self.fc_valence = nn.Linear(self._feat_dim, 2)
        self.fc_arousal = nn.Linear(self._feat_dim, 2)

    def forward(self, x):
        x = self.firstconv(x)
        x = self.depthwise(x)
        x = self.separable(x)

        x = x.reshape(x.size(0), -1)

        v = self.fc_valence(x)
        a = self.fc_arousal(x)
        return v, a


In [26]:
def mixup_data(x, y, alpha=0.2):
    """MixUp for 2-label (valence, arousal) classification."""
    if alpha <= 0:
        return x, y, 1.0  # no mixup
    
    batch_size = x.size(0)
    index = torch.randperm(batch_size)

    lam = np.random.beta(alpha, alpha)

    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]  # (B,2), each column is a class index

    return mixed_x, (y_a, y_b), lam


In [27]:
def compute_class_weights(labels_bin):
    """
    labels_bin: Nx2 {0,1}
    returns:
        val_weight, aro_weight (torch FloatTensor of shape (2,))
    """

    val_counts = np.bincount(labels_bin[:, 0])
    aro_counts = np.bincount(labels_bin[:, 1])

    val_weight = torch.tensor(1.0 / val_counts, dtype=torch.float32)
    aro_weight = torch.tensor(1.0 / aro_counts, dtype=torch.float32)

    # Normalize so max weight = 1
    val_weight = val_weight / val_weight.max()
    aro_weight = aro_weight / aro_weight.max()

    return val_weight, aro_weight


In [28]:
def va_binary_accuracy(preds, labels):
    """
    preds: (B, 2) continuous outputs
    labels: (B, 2) continuous labels
    Returns: val_acc, aro_acc, avg_acc
    """

    # predicted bins
    pred_val = (preds[:, 0] >= 0).long()
    pred_aro = (preds[:, 1] >= 0).long()

    # true bins
    true_val = (labels[:, 0] >= 0).long()
    true_aro = (labels[:, 1] >= 0).long()

    val_acc = (pred_val == true_val).float().mean().item()
    aro_acc = (pred_aro == true_aro).float().mean().item()

    return val_acc, aro_acc, (val_acc + aro_acc) / 2


In [85]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


def train_eegnet_classification(
    model,
    train_loader,
    val_loader,
    val_weight,
    aro_weight,
    batch_size=64,
    dropout=0.5,
    n_epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
    mixup_alpha=0.05
):
    print(f"Training on {device}...")

    model = model.to(device)

    # Loss functions (with class-balance + smoothing)
    criterion_val = nn.CrossEntropyLoss(
        weight=val_weight.to(device),
        #label_smoothing=0.1
    )
    criterion_aro = nn.CrossEntropyLoss(
        weight=aro_weight.to(device),
        #label_smoothing=0.1
    )

    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, factor=0.5, patience=5, min_lr=1e-6
    )

    best_val_loss = float("inf")
    best_path = f"best_eegnet_classification.pth"

    for epoch in range(1, n_epochs + 1):

        # ----------------------------- TRAINING -----------------------------
        model.train()
        train_loss = 0
        total = 0
        v_correct = 0
        a_correct = 0

        for X, y in tqdm(train_loader, desc=f"[Train {epoch}]", leave=False):

            X, y = X.to(device), y.to(device)

            # ------- MixUp -------
            
            #X_mix, (y_a, y_b), lam = mixup_data(X, y, alpha=mixup_alpha)

            #val_y_a = y_a[:, 0]
            #aro_y_a = y_a[:, 1]
            #val_y_b = y_b[:, 0]
            #aro_y_b = y_b[:, 1]

            optimizer.zero_grad()
            logits = model(X)
            loss = criterion_val(logits[0], y[:, 0]) + criterion_aro(logits[1], y[:, 1])
            v_pred, a_pred = model(X)

            # MixUp loss
            #loss_v = lam * criterion_val(v_pred, val_y_a) + (1 - lam) * criterion_val(v_pred, val_y_b)
            #loss_a = lam * criterion_aro(a_pred, aro_y_a) + (1 - lam) * criterion_aro(a_pred, aro_y_b)
            #loss = loss_v + loss_a

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * X.size(0)
            total += X.size(0)

            # classification accuracy (NO mixup for accuracy)
            v_hat = v_pred.argmax(dim=1)
            a_hat = a_pred.argmax(dim=1)

            v_correct += (v_hat == y[:, 0]).sum().item()
            a_correct += (a_hat == y[:, 1]).sum().item()

        train_loss /= total
        train_v_acc = v_correct / total
        train_a_acc = a_correct / total
        train_avg_acc = (train_v_acc + train_a_acc) / 2

        # ----------------------------- VALIDATION -----------------------------
        model.eval()
        val_loss = 0
        total = 0
        v_correct = 0
        a_correct = 0

        with torch.no_grad():
            for X, y in tqdm(val_loader, desc=f"[Val {epoch}]", leave=False):
                X, y = X.to(device), y.to(device)

                val_y = y[:, 0]
                aro_y = y[:, 1]

                v_pred, a_pred = model(X)

                loss_v = criterion_val(v_pred, val_y)
                loss_a = criterion_aro(a_pred, aro_y)
                loss = loss_v + loss_a

                val_loss += loss.item() * X.size(0)
                total += X.size(0)

                v_hat = v_pred.argmax(dim=1)
                a_hat = a_pred.argmax(dim=1)

                v_correct += (v_hat == val_y).sum().item()
                a_correct += (a_hat == aro_y).sum().item()

        val_loss /= total
        val_v_acc = v_correct / total
        val_a_acc = a_correct / total
        val_avg_acc = (val_v_acc + val_a_acc) / 2

        # ----------------------------- LOGGING -----------------------------
        print(
            f"Epoch {epoch}/{n_epochs} | "
            f"Train Loss: {train_loss:.4f} | Train Acc: {train_avg_acc:.4f} | "
            f"Val Loss: {val_loss:.4f} | Val Acc: {val_avg_acc:.4f}"
        )

        scheduler.step(val_loss)

        # Save best model
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), best_path)
            print("🔥 Saved best model!")

    print("Training Completed!")
    print(f"Best val loss = {best_val_loss:.4f}")
    model_name = f"eegnet_final_{time.strftime('%Y%m%d_%H%M%S')}_lr{lr}_bs{batch_size}_dropout{dropout}_classification_wd{weight_decay}_alpha{mixup_alpha}_ep{n_epochs}_final.pth"
    save_path= f"C:/Users/Yebbet/Desktop/School/APS360/EEG_Datasets/weights_eegnet/{model_name}"
    torch.save(model.state_dict(), save_path)


In [127]:
val_weight, aro_weight = compute_class_weights(all_labels_bin)
model = EEGNet(
    n_channels=14,
    n_samples=512,        
    F1=8,
    D=2,
    F2=16,
    kernel_length=64,
    pool_size=4,
    dropout=0.6,
)
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=128, shuffle=False)
train_eegnet_classification(
    model,
    train_loader,
    val_loader,
    val_weight,
    aro_weight,
    n_epochs=50,
    lr=5e-3,
    mixup_alpha=0.05
)


Training on cpu...


Epoch 1/50 | Train Loss: 1.4785 | Train Acc: 0.5383 | Val Loss: 1.3875 | Val Acc: 0.4872
🔥 Saved best model!


Epoch 2/50 | Train Loss: 1.4391 | Train Acc: 0.5383 | Val Loss: 1.4070 | Val Acc: 0.4423


Epoch 3/50 | Train Loss: 1.4000 | Train Acc: 0.5311 | Val Loss: 1.4330 | Val Acc: 0.4551


Epoch 4/50 | Train Loss: 1.3583 | Train Acc: 0.5873 | Val Loss: 1.4617 | Val Acc: 0.4679


...

Epoch 46/50 | Train Loss: 1.1535 | Train Acc: 0.6722 | Val Loss: 1.5552 | Val Acc: 0.5449


Epoch 47/50 | Train Loss: 1.1952 | Train Acc: 0.6627 | Val Loss: 1.5546 | Val Acc: 0.5449


Epoch 48/50 | Train Loss: 1.1870 | Train Acc: 0.6866 | Val Loss: 1.5537 | Val Acc: 0.5449


Epoch 49/50 | Train Loss: 1.2286 | Train Acc: 0.6591 | Val Loss: 1.5549 | Val Acc: 0.5513


Epoch 50/50 | Train Loss: 1.1966 | Train Acc: 0.6639 | Val Loss: 1.5556 | Val Acc: 0.5513
Training Completed!
Best val loss = 1.3875
